In [15]:
from collections.abc import Callable
from pathlib import Path
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import seaborn as sns
import sys
sys.path.insert(0, str(Path.cwd().parent))


from src.encoder import DummyEncoder
from src import quant, search



In [16]:
data_dir = Path("../data")
chunks = pd.read_csv(data_dir / "chunks.csv")
questions = pd.read_csv(data_dir / "questions.csv")
answer_chunks = pd.read_csv(data_dir / "answer_chunks.csv")

chunks.head()

,chunk_id,pid,chunk_no,start,end,text
0,0,0,0,0,127,"Architecturally, the school has a Catholic cha..."
1,1,0,1,54,270,Atop the Main Building's gold dome is a golden...
2,2,0,2,128,333,Immediately in front of the Main Building and ...
3,3,0,3,271,421,Next to the Main Building is the Basilica of t...
4,4,0,4,334,550,"Immediately behind the basilica is the Grotto,..."


In [17]:
model_name = "BAAI/bge-small-en-v1.5"
save_dir = data_dir / f"embeddings_{model_name.rsplit('/', 1)[-1]}"

chunk_ids = np.load(save_dir / "chunk_ids.npy")
chunk_embeddings = np.load(save_dir / "chunk_embeddings.npy")
query_ids = np.load(save_dir / "query_ids.npy")
query_embeddings = np.load(save_dir / "query_embeddings.npy")

chunks = chunks.set_index("chunk_id", drop=False).loc[chunk_ids].reset_index(drop=True)
questions = questions.set_index("qid", drop=False).loc[query_ids].reset_index(drop=True)


In [18]:
encoder = DummyEncoder(
    chunk_texts=chunks["text"].tolist(),
    chunk_embeddings=chunk_embeddings,
    query_texts=questions["question"].tolist(),
    query_embeddings=query_embeddings,
)


In [19]:
def hit(retrieved: set[int], relevant: set[int]) -> float:
    """Есть ли среди найденных фрагментов хотя бы один эталонный."""
    return float(len(retrieved & relevant) > 0)


def precision(retrieved: set[int], relevant: set[int]) -> float:
    """Доля эталонных фрагментов среди найденных."""
    return len(retrieved & relevant) / len(retrieved)


def recall(retrieved: set[int], relevant: set[int]) -> float:
    """Доля найденных фрагментов среди всех эталонных."""
    return len(retrieved & relevant) / len(relevant)


def calculate_metrics(
    top_ids: np.ndarray,
    questions: pd.DataFrame,
    answer_chunks: pd.DataFrame,
    metric_fn: Callable[[set[int], set[int]], float],
    ks: tuple[int,...] = (1, 5, 10),
) -> pd.DataFrame:
    """Усредняет выбранную метрику по строгому и мягкому эталонам."""
    metrics = {}
    for name, gold in (
        ("strict", answer_chunks[answer_chunks["is_annotated"]]),
        ("soft", answer_chunks),
    ):
        gold_chunks = gold.groupby("orig_qid")["chunk_id"].agg(set).to_dict()
        metrics[name] = []
        for k in ks:
            values = [
                metric_fn(set(row[:k]), gold_chunks[orig_qid])
                for orig_qid, row in zip(questions["orig_qid"], top_ids)
            ]
            metrics[name].append(np.mean(values))

    return pd.DataFrame(metrics, index=pd.Index(ks, name="k"))

In [20]:
def evaluate_search(
    chunks: pd.DataFrame,
    questions: pd.DataFrame,
    answer_chunks: pd.DataFrame,
    metric_fn: Callable[[set[int], set[int]], float],
    searcher: search.BaseSearch,
    ks: tuple[int, ...] = (1, 5, 10),
) -> pd.DataFrame:
    """Обучает поиск на train-текстах и оценивает поиск по всей базе.

    Метрики train/valid и исходных вопросов/перефразов считаются отдельно.
    """
    train_pids = questions.loc[questions["role"] == "train", "pid"]
    chunk_train_mask = chunks["pid"].isin(train_pids)
    searcher.fit(chunks.loc[chunk_train_mask, "text"].tolist())
    searcher.add(chunks["text"].tolist(), chunks["chunk_id"].to_numpy())
    top_ids, _ = searcher.search(questions["question"].tolist(), k=max(ks))

    results = []
    for role in ("train", "valid"):
        for question_type, is_paraphrase in (("real", False), ("paraphrase", True)):
            mask = (
                (questions["role"] == role)
                & (questions["is_paraphrase"] == is_paraphrase)
            ).to_numpy()
            group_metrics = calculate_metrics(
                top_ids[mask], questions.loc[mask], answer_chunks, metric_fn, ks,
            )
            group_metrics.columns = [
                f"{role}_{name}_{question_type}" for name in group_metrics.columns
            ]
            results.append(group_metrics)
    return pd.concat(results, axis=1)


In [21]:
baseline_metrics = evaluate_search(
    chunks=chunks,
    questions=questions,
    answer_chunks=answer_chunks,
    metric_fn=recall,
    searcher=search.BaseSearch(encoder=encoder, quantizer=quant.BaseQuant()),
    ks=(1, 5, 10, 15, 20, 25),
)
baseline_metrics

,train_strict_real,train_soft_real,train_strict_paraphrase,train_soft_paraphrase,valid_strict_real,valid_soft_real,valid_strict_paraphrase,valid_soft_paraphrase
k,,,,,,,,
1,0.469188,0.451656,0.34225,0.329042,0.460404,0.436215,0.324667,0.318038
5,0.769139,0.748778,0.61750,0.599058,0.766831,0.737614,0.601833,0.590627
10,0.828908,0.810351,0.70575,0.686250,0.829147,0.802350,0.688833,0.671449
15,0.856283,0.838851,0.74475,0.726892,0.856895,0.831816,0.729333,0.708613
20,0.873499,0.856897,0.77550,0.757925,0.872157,0.848781,0.761000,0.740668
25,0.885942,0.870032,0.79475,0.777583,0.886224,0.864467,0.779333,0.760124


In [22]:
ivf_metrics = evaluate_search(
    chunks=chunks,
    questions=questions,
    answer_chunks=answer_chunks,
    metric_fn=recall,
    searcher=search.IvfSearch(
        encoder=encoder,
        quantizer=quant.BaseQuant(),
        nlist=1024,
        nprobe=16,
    ),
    ks=(1, 5, 10, 15, 20, 25),
)
ivf_metrics

,train_strict_real,train_soft_real,train_strict_paraphrase,train_soft_paraphrase,valid_strict_real,valid_soft_real,valid_strict_paraphrase,valid_soft_paraphrase
k,,,,,,,,
1,0.432583,0.416547,0.31425,0.303358,0.417034,0.394652,0.284167,0.279186
5,0.697669,0.679543,0.56550,0.550042,0.672662,0.646903,0.528833,0.519208
10,0.748008,0.731830,0.64250,0.626625,0.724415,0.700650,0.598167,0.584530
15,0.770783,0.755792,0.67725,0.662392,0.745687,0.724093,0.627667,0.613627
20,0.784636,0.770604,0.70400,0.688608,0.757247,0.736879,0.654500,0.640683
25,0.794346,0.781062,0.71875,0.704308,0.767539,0.748535,0.674167,0.661638


In [23]:
ivf_rabitq_metrics = evaluate_search(
    chunks=chunks,
    questions=questions,
    answer_chunks=answer_chunks,
    metric_fn=recall,
    searcher=search.IvfRaBitqSearch(
        encoder=encoder,
        quantizer=quant.BaseQuant(),
        nlist=1024,
        nprobe=16,
    ),
    ks=(1, 5, 10, 15, 20, 25),
)
ivf_rabitq_metrics

,train_strict_real,train_soft_real,train_strict_paraphrase,train_soft_paraphrase,valid_strict_real,valid_soft_real,valid_strict_paraphrase,valid_soft_paraphrase
k,,,,,,,,
1,0.432565,0.416532,0.31425,0.303358,0.417129,0.394746,0.285167,0.280186
5,0.697680,0.679555,0.56550,0.550042,0.672756,0.646997,0.528833,0.519208
10,0.748002,0.731825,0.64250,0.626625,0.724368,0.700602,0.598167,0.584530
15,0.770777,0.755788,0.67775,0.662892,0.745640,0.724046,0.625667,0.611627
20,0.784522,0.770509,0.70400,0.688608,0.757105,0.736800,0.652500,0.638683
25,0.794174,0.780904,0.71875,0.704308,0.767822,0.748788,0.672167,0.659638


In [24]:
ivf_rabitq_metrics = evaluate_search(
    chunks=chunks,
    questions=questions,
    answer_chunks=answer_chunks,
    metric_fn=recall,
    searcher=search.HnswSearch(encoder=encoder, quantizer=quant.BaseQuant()),
    ks=(1, 5, 10, 15, 20, 25),
)
ivf_rabitq_metrics

,train_strict_real,train_soft_real,train_strict_paraphrase,train_soft_paraphrase,valid_strict_real,valid_soft_real,valid_strict_paraphrase,valid_soft_paraphrase
k,,,,,,,,
1,0.462201,0.444891,0.33475,0.322458,0.453403,0.429364,0.319667,0.313038
5,0.754247,0.734227,0.60475,0.587308,0.753073,0.724113,0.593833,0.582827
10,0.811561,0.793343,0.68975,0.671558,0.814207,0.787898,0.675833,0.658716
15,0.837441,0.820415,0.73050,0.713517,0.840016,0.815556,0.708333,0.689313
20,0.853475,0.837326,0.76025,0.743675,0.854836,0.832059,0.740000,0.720868
25,0.865188,0.849716,0.77900,0.762708,0.867783,0.846495,0.759833,0.742224


### BM25 и гибридный поиск

In [25]:
paraphrase_ids = questions.loc[questions["is_paraphrase"], "orig_qid"]
paired_mask = questions["orig_qid"].isin(paraphrase_ids).to_numpy()
paired_questions = questions.loc[paired_mask]


In [26]:
bm25_metrics = evaluate_search(
    chunks=chunks,
    questions=paired_questions,
    answer_chunks=answer_chunks,
    metric_fn=recall,
    searcher=search.Bm25Search(encoder=encoder, quantizer=quant.BaseQuant()),
    ks=(1, 5, 10, 15, 20, 25),
)
bm25_metrics


,train_strict_real,train_soft_real,train_strict_paraphrase,train_soft_paraphrase,valid_strict_real,valid_soft_real,valid_strict_paraphrase,valid_soft_paraphrase
k,,,,,,,,
1,0.45950,0.440558,0.14700,0.140175,0.503667,0.483541,0.171500,0.171167
5,0.73200,0.705621,0.29375,0.283117,0.791333,0.759530,0.324167,0.320967
10,0.78475,0.758346,0.35075,0.338442,0.831000,0.797530,0.383667,0.378252
15,0.80625,0.780005,0.38625,0.373500,0.854167,0.822486,0.412833,0.406486
20,0.82550,0.799546,0.41400,0.400600,0.866000,0.835319,0.440833,0.432652
25,0.83625,0.810855,0.43400,0.420508,0.870000,0.840386,0.473333,0.465552


In [27]:
hybrid_metrics = evaluate_search(
    chunks=chunks,
    questions=paired_questions,
    answer_chunks=answer_chunks,
    metric_fn=recall,
    searcher=search.HybridBm25Search(
        vector_search=search.IvfSearch(
            encoder=encoder,
            quantizer=quant.BaseQuant(),
        ),
        alpha=0.5,
        num_candidates=100,
    ),
    ks=(1, 5, 10, 15, 20, 25),
)
hybrid_metrics


,train_strict_real,train_soft_real,train_strict_paraphrase,train_soft_paraphrase,valid_strict_real,valid_soft_real,valid_strict_paraphrase,valid_soft_paraphrase
k,,,,,,,,
1,0.52500,0.503958,0.26800,0.256450,0.491833,0.466694,0.238333,0.234952
5,0.82250,0.796805,0.54800,0.530975,0.822667,0.791887,0.523333,0.516005
10,0.87000,0.847163,0.65075,0.632567,0.873833,0.846687,0.618500,0.610527
15,0.89275,0.870296,0.70775,0.688408,0.895333,0.869921,0.659000,0.645105
20,0.90725,0.886721,0.74475,0.725467,0.904833,0.881710,0.683500,0.669060
25,0.91475,0.895546,0.76500,0.746392,0.913833,0.891821,0.702167,0.687616


In [28]:
bm25_metrics = evaluate_search(
    chunks=chunks,
    questions=paired_questions,
    answer_chunks=answer_chunks,
    metric_fn=recall,
    searcher=search.IvfSearch(encoder=encoder, quantizer=quant.BaseQuant()),
    ks=(1, 5, 10, 15, 20, 25),
)
bm25_metrics


,train_strict_real,train_soft_real,train_strict_paraphrase,train_soft_paraphrase,valid_strict_real,valid_soft_real,valid_strict_paraphrase,valid_soft_paraphrase
k,,,,,,,,
1,0.47000,0.450600,0.33775,0.324642,0.438833,0.418594,0.296833,0.291471
5,0.74900,0.726067,0.60675,0.588308,0.669167,0.649854,0.528667,0.519527
10,0.80225,0.783367,0.69150,0.672625,0.722833,0.702210,0.593000,0.581416
15,0.82475,0.807108,0.72750,0.710617,0.737167,0.721565,0.616167,0.603279
20,0.83850,0.821596,0.75675,0.739833,0.754667,0.739176,0.642500,0.629835
25,0.85400,0.837488,0.77375,0.757100,0.759667,0.745621,0.656500,0.645068
